# DDPG Training Notebook for HVAC DRL
Compatible with Google Colab, Kaggle, and Local environments. This notebook reproduces the training script `paper_reference/train.py` and saves checkpoints to `checkpoints_v2/`.

In [ ]:
# Setup environment and paths
import os
import sys

IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
IN_COLAB = 'google.colab' in sys.modules

print(f"IN_KAGGLE: {IN_KAGGLE}")
print(f"IN_COLAB: {IN_COLAB}")

# Clone repository if running in Colab/Kaggle environments without the repo files
if (IN_KAGGLE or IN_COLAB) and not os.path.exists('simulator'):
    print("Cloning repository to access simulator and agent modules...")
    repo_url = "https://github.com/trandat09062003/Tem_Humid_C02_Pm2.5-Smart-HVAC-control.git"
    repo_name = "Tem_Humid_C02_Pm2.5-Smart-HVAC-control"
    
    if not os.path.exists(repo_name):
        !git clone {repo_url}
    
    # Check if original_hvac or paper_reference exists in the cloned repository
    hvac_dir = os.path.join(repo_name, 'original_hvac')
    if not os.path.exists(hvac_dir):
        hvac_dir = os.path.join(repo_name, 'paper_reference')
        
    repo_path = os.path.abspath(hvac_dir)
    if repo_path not in sys.path:
        sys.path.append(repo_path)
    os.chdir(repo_path)

print(f"Current Working Directory: {os.getcwd()}")

In [ ]:
# GPU configuration & dependency install
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(gpus))

if len(gpus) == 0:
    print("\n" + "="*80)
    print("WARNING: No GPU detected! Training DDPG for 5000 episodes on CPU will be extremely slow.")
    print("Please enable the GPU accelerator in your Kaggle Notebook settings:")
    print("1. In the right-hand panel of your Kaggle Notebook editor, expand the 'Settings' section.")
    print("2. Under 'Accelerator', change 'None' to 'GPU T4 x2' or 'GPU P100'.")
    print("3. Restart the session (or it will restart automatically) and run all cells again.")
    print("="*80 + "\n")
else:
    print("GPU Details:")
    for gpu in gpus:
        print("  -", gpu)

if IN_COLAB:
    print("Installing dependencies on Colab...")
    !pip install -q tensorflow==2.16 numpy matplotlib tqdm
elif not IN_KAGGLE:
    print("Running locally. Please ensure tensorflow, numpy, matplotlib, and tqdm are installed.")

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from simulator.hybrid_sim import HybridSimulator
from drl.ddpg_agent import DDPGAgentV2
from data.weather_gen import SeoulWeatherGenerator

In [ ]:
# Hyper‑parameters (same as train.py)
STATE_MIN = np.array([0,-5,0.002,0,390,0,15,0.003,400,0], dtype=np.float32)
STATE_MAX = np.array([24,40,0.025,900,510,80,35,0.022,2000,50], dtype=np.float32)
def norm(s): return (np.array(s, dtype=np.float32)-STATE_MIN)/(STATE_MAX-STATE_MIN+1e-8)
def ddpg2sim(a): return (np.clip(a,-1,1)+1)/2
MONTHS = [5,6,7,8,9,10]
N_EPISODES = 5000

In [ ]:
# Initialise components
sim = HybridSimulator()
agent = DDPGAgentV2()
weather = SeoulWeatherGenerator(seed=42)
rewards = []

In [ ]:
def run_episode(sim, agent, weather, train=True):
    total_r, steps = 0.0, 0
    for month in MONTHS:
        for _ in range(30):
            T_d, om_d, qs_d, pm_d = weather.generate_day(month)
            state = np.array([0,T_d[0],om_d[0],qs_d[0],450,pm_d[0],24,0.010,600,5], dtype=np.float32)
            for step in range(96):
                state[0]=step*0.25; state[1]=T_d[step]; state[2]=om_d[step]
                state[3]=qs_d[step]; state[5]=pm_d[step]
                sn = norm(state)
                a_ddpg = agent.select_action(sn, add_noise=train)
                a_sim = ddpg2sim(a_ddpg)
                next_s, reward, _ = sim.step(state.tolist(), a_sim)
                next_s = np.array(next_s, dtype=np.float32)
                if train:
                    agent.store(sn, a_ddpg, reward, norm(next_s))
                    agent.train_step()
                state = next_s
                total_r += reward; steps += 1
    return total_r / steps

In [ ]:
# Training loop with progress bar
for ep in tqdm(range(1, N_EPISODES+1), desc='Training'):
    agent.noise.reset()
    avg_r = run_episode(sim, agent, weather, train=True)
    rewards.append(avg_r)
    if ep % 5 == 0:
        os.makedirs('checkpoints_v2', exist_ok=True)
        agent.save('checkpoints_v2')

In [ ]:
# Plot training curve
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(rewards, color='steelblue', label='DDPG Training')
ax.set_xlabel('Episode')
ax.set_ylabel('Avg Reward / step')
ax.set_title('DDPG Training Curve')
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Save and zip checkpoints for download
import shutil
checkpoint_dir = 'checkpoints_v2'
if os.path.exists(checkpoint_dir):
    shutil.make_archive('checkpoints_v2', 'zip', checkpoint_dir)
    print("Successfully zipped checkpoints to checkpoints_v2.zip")